# 🎯 Module 8.2: Hyperparameter Tuning with Optuna + MLFlow

**Goal**: Use Optuna for intelligent hyperparameter optimization with full MLFlow tracking.

---

In [ ]:
import mlflow
import optuna
from optuna.integration.mlflow import MLflowCallback
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings("ignore")

mlflow.autolog(disable=True)
mlflow.set_experiment("08_hyperparameter_tuning")

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Setup complete!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 1. Manual Optuna + MLFlow Integration

Log each Optuna trial as a nested MLFlow run.

In [ ]:
def objective(trial):
    """Optuna objective function — each call is one trial."""
    
    # Suggest hyperparameters
    n_estimators = trial.suggest_int("n_estimators", 50, 300, step=50)
    max_depth = trial.suggest_int("max_depth", 2, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    
    # Log each trial as a nested run
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        params = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_split": min_samples_split,
            "min_samples_leaf": min_samples_leaf,
        }
        mlflow.log_params(params)
        
        # Train and evaluate with cross-validation
        model = RandomForestClassifier(**params, random_state=42)
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
        
        mean_cv_accuracy = cv_scores.mean()
        std_cv_accuracy = cv_scores.std()
        
        # Also evaluate on test set
        model.fit(X_train, y_train)
        test_accuracy = accuracy_score(y_test, model.predict(X_test))
        
        mlflow.log_metrics({
            "cv_accuracy_mean": mean_cv_accuracy,
            "cv_accuracy_std": std_cv_accuracy,
            "test_accuracy": test_accuracy,
        })
        mlflow.set_tag("trial_number", trial.number)
    
    return mean_cv_accuracy  # Optuna optimizes this

# Run the optimization inside a parent run
with mlflow.start_run(run_name="optuna_hpo_study"):
    mlflow.set_tag("purpose", "hyperparameter_optimization")
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_trials", 30)
    
    # Create and run Optuna study
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=30, show_progress_bar=True)
    
    # Log best results on parent
    mlflow.log_metric("best_cv_accuracy", study.best_value)
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
    
    print(f"\n🏆 Best trial: #{study.best_trial.number}")
    print(f"   CV Accuracy: {study.best_value:.4f}")
    print(f"   Params: {study.best_params}")

## 2. Visualize Optimization History

In [ ]:
import matplotlib.pyplot as plt

# Plot optimization history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trials_df = study.trials_dataframe()
axes[0].plot(trials_df["number"], trials_df["value"], marker='o', markersize=4, alpha=0.7)
axes[0].axhline(y=study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.4f}')
axes[0].set_xlabel("Trial Number")
axes[0].set_ylabel("CV Accuracy")
axes[0].set_title("Optimization History")
axes[0].legend()

# Parameter importances
importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()), color='steelblue')
axes[1].set_xlabel("Importance")
axes[1].set_title("Hyperparameter Importance")

plt.tight_layout()
plt.show()

## 3. Train Final Model with Best Params

In [ ]:
with mlflow.start_run(run_name="best_model_from_hpo"):
    best_model = RandomForestClassifier(**study.best_params, random_state=42)
    best_model.fit(X_train, y_train)
    
    test_acc = accuracy_score(y_test, best_model.predict(X_test))
    
    mlflow.log_params(study.best_params)
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.set_tag("source", "optuna_hpo")
    
    from mlflow.models import infer_signature
    sig = infer_signature(X_train, best_model.predict(X_train))
    mlflow.sklearn.log_model(best_model, "model", signature=sig)
    
    print(f"✅ Best model trained and logged!")
    print(f"   Test accuracy: {test_acc:.4f}")
    print(f"   Params: {study.best_params}")

## 🔑 Key Takeaways

1. **Optuna** provides intelligent HPO (not just random/grid search)
2. Log each trial as a **nested run** under a parent HPO run
3. Use **cross-validation** as the objective for more robust optimization
4. **Optuna's importance analysis** shows which hyperparameters matter most
5. After HPO, train a **final model** with the best params and register it

---

## ➡️ Next: `03_pipeline_tracking.ipynb`